# LUMIS-1 Speaker Fine-Tuning: Granite 4.0-1B (Dense)

**Target:** RunPod PyTorch 2.4.0 Template (NVIDIA A40/A100, CUDA 12.x)

This notebook fine-tunes the **Dense** `ibm-granite/granite-4.0-1b` model on the Lumis-1 speaker dataset.
It uses LoRA for efficient training and merges the adapter for export.

## ⚠️ CRITICAL PRE-REQUISITES
1. **Upload Data**: You must upload the `dataset` folder (containing `train.jsonl` and `eval.jsonl`) to this directory.
2. **GPU**: A generic A40 (48GB) or A100 (80GB) is recommended, though 1B models can run on smaller GPUs (24GB).


In [ ]:
# ============================================================================
# CELL 1: ROBUST DATA VERIFICATION
# ============================================================================
import os
import glob

print("=" * 60)
print("[LUMIS-1] Verifying Dataset Existence")
print("=" * 60)

# Define potential paths for robustness
potential_paths = [
    "dataset",
    "./dataset",
    "../dataset",
    "/workspace/dataset"
]

train_file = None
eval_file = None

for base in potential_paths:
    t = os.path.join(base, "train.jsonl")
    e = os.path.join(base, "eval.jsonl")
    if os.path.exists(t) and os.path.exists(e):
        train_file = t
        eval_file = e
        break

if not train_file:
    print("\n!!! ERROR: DATASET NOT FOUND !!!")
    print("Scanned paths:", potential_paths)
    print("Please upload the 'dataset' folder containing 'train.jsonl' and 'eval.jsonl'.")
    # Stop execution intentionally if missing
    raise FileNotFoundError("Dataset missing. Upload 'dataset/train.jsonl' and 'dataset/eval.jsonl' first.")

print(f"\u2705 Found TRAIN: {train_file} ({os.path.getsize(train_file)/1024/1024:.2f} MB)")
print(f"\u2705 Found EVAL:  {eval_file} ({os.path.getsize(eval_file)/1024/1024:.2f} MB)")

In [ ]:
# ============================================================================
# CELL 2: SETUP & DEPENDENCIES
# ============================================================================
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("[LUMIS-1] Installing Dependencies...")
# Install specific versions known to work for Granite/Llama
pkgs = [
    "transformers>=4.45.0",
    "datasets>=2.18.0",
    "accelerate>=0.28.0",
    "peft>=0.13.0",
    "trl>=0.12.0",
    "bitsandbytes>=0.43.0"
]

for p in pkgs:
    print(f"  - Installing {p}...")
    install(p)

import torch
print(f"\n[SYSTEM] Torch: {torch.__version__}, CUDA: {torch.version.cuda}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this notebook!")

In [ ]:
# ============================================================================
# CELL 3: LOAD MODEL (DENSE)
# ============================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

# CORRECT MODEL ID FOR DENSE GRANITE
MODEL_ID = "ibm-granite/granite-4.0-1b"

print(f"[LUMIS-1] Loading DENSE Model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("\u2705 Model Loaded. Architecture:", model.config.architectures)
# Critic: Ensure it is CausalLM and not Mamba (check config type)
print("Config Type:", model.config.model_type)

In [ ]:
# ============================================================================
# CELL 4: LORA CONFIG
# ============================================================================
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ============================================================================
# CELL 5: DATASET PREP & TRAINING
# ============================================================================
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# Load
ds_train = load_dataset("json", data_files=train_file, split="train")
ds_eval = load_dataset("json", data_files=eval_file, split="train")

print(f"Training Details: {len(ds_train)} examples, {len(ds_eval)} eval examples")

output_dir = "./lumis1-speaker-v1"

args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective Batch = 16
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_seq_length=2048,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    dataset_text_field="text",
    packing=False
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer
)

print("\u2705 Starting Training...")
trainer.train()
trainer.save_model(output_dir)
print("\u2705 Training Complete. LoRA saved.")

In [ ]:
# ============================================================================
# CELL 6: MERGE & EXPORT (PYTORCH)
# ============================================================================
from peft import PeftModel
import torch

print("[LUMIS-1] Merging Adapter...")

# 1. Free memory
del model
del trainer
torch.cuda.empty_cache()

# 2. Reload Base
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  # Float16 for export compatibility
    device_map="auto"
)

# 3. Merge
model = PeftModel.from_pretrained(base_model, output_dir)
model = model.merge_and_unload()

# 4. Save
final_path = "./lumis1-speaker-v1-merged"
model.save_pretrained(final_path, safe_serialization=True)
tokenizer.save_pretrained(final_path)

print(f"\u2705 Merged Model Saved to: {final_path}")